# Exploratory Data Analysis (EDA)
## Objective
Explore the dataset to uncover patterns, identify data quality issues, and form hypotheses to guide feature engineering.

### Instructions:
1. Overview of the Data
2. Summary Statistics
3. Distribution of Numerical Features
4. Distribution of Categorical Features
5. Correlation Analysis
6. Identifying Missing Values
7. Outlier Detection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Overview of the Data
Understand the structure of the dataset, including the number of rows, columns, and data types.

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('../data/raw/data.csv')
    print("Dataset loaded successfully.")
    print(f"Shape: {df.shape}")
    display(df.head())
    display(df.info())
except FileNotFoundError:
    print("Error: data/raw/data.csv not found. Please ensure the dataset is in the correct directory.")

## 2. Summary Statistics
Understand the central tendency, dispersion, and shape of the dataset’s distribution.

In [ ]:
if 'df' in locals():
    display(df.describe(include='all'))
    print(f"\nUnique customers: {df['CustomerId'].nunique()}")
    print(f"Date range: {df['TransactionStartTime'].min()} to {df['TransactionStartTime'].max()}")
    skew = df.select_dtypes(include=[np.number]).skew()
    print("\nSkewness of numerical features:")
    display(skew)

## 3. Distribution of Numerical Features
Visualize the distribution of numerical features to identify patterns, skewness, and potential outliers.

In [ ]:
if 'df' in locals():
    num_features = df.select_dtypes(include=[np.number]).columns
    for col in num_features:
        plt.figure(figsize=(10, 5))
        sns.histplot(df[col], kde=True)
        plt.title(f'Distribution of {col}')
        plt.show()

## 4. Distribution of Categorical Features
Analyzing the distribution of categorical features provides insights into the frequency and variability of categories.

In [ ]:
if 'df' in locals():
    cat_features = df.select_dtypes(include=['object']).columns
    for col in cat_features:
        if df[col].nunique() < 20:  # Only plot if number of categories is manageable
            plt.figure(figsize=(10, 5))
            sns.countplot(x=col, data=df)
            plt.title(f'Distribution of {col}')
            plt.xticks(rotation=45)
            plt.show()

## 5. Correlation Analysis
Understanding the relationship between numerical features.

In [ ]:
if 'df' in locals():
    plt.figure(figsize=(12, 8))
    sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix')
    plt.show()

## 6. Identifying Missing Values
Identify missing values to determine missing data and decide on appropriate imputation strategies.

In [ ]:
if 'df' in locals():
    missing_values = df.isnull().sum()
    missing_percent = (missing_values / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percent})
    if missing_df['Missing Values'].sum() == 0:
        print("No missing values detected in any column.")
    display(missing_df[missing_df['Missing Values'] > 0].sort_values(by='Percentage', ascending=False))

## 7. Outlier Detection
Use box plots to identify outliers.

In [ ]:
if 'df' in locals():
    for col in num_features:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[col])
        plt.title(f'Box Plot of {col}')
        plt.show()

## Summary of Insights

Based on the exploratory analysis above, the following are the **top insights** to guide feature engineering and modeling:

1. **Transaction-level granularity with few customers:** The dataset records individual mobile-money transactions (`TransactionId`) linked to a small number of unique `CustomerId` values. Modeling should aggregate transaction-level rows into **customer-level features** (total/average amount, transaction count, etc.) before training a credit-risk model.

2. **No missing values in core fields:** All 14 columns are fully populated in this sample. Imputation steps should still be included in the production pipeline for robustness, but exploratory checks suggest **median/mode imputation** is a reasonable default rather than row removal.

3. **Highly skewed and signed transaction amounts:** `Amount` ranges from negative values (likely reversals or credits) to large positive transfers (up to 20,000), with high variance relative to the mean. **Log transforms, outlier capping, or standardization** will help stabilize numerical features; negative amounts may signal disengagement or dispute behavior worth tracking per customer.

4. **Dominant product and channel categories:** `ProductCategory` is dominated by **airtime** and **utility_bill** purchases, while `ChannelId_3` accounts for most transactions. These low-cardinality categoricals are good candidates for **one-hot encoding** and **WoE transformation** against a proxy default label.

5. **Strong Amount–Value relationship:** The correlation heatmap shows `Amount` and `Value` move together closely. One of these may be redundant after aggregation; consider keeping aggregated amount statistics and monitoring **Information Value (IV)** when selecting features for the scorecard.